# Lab-5

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.8 MB/s eta 0:00:00


In [3]:
from docx import Document

In [5]:
doc = Document("dataset.docx")
data = " ".join([p.text for p in doc.paragraphs]).lower()

Opens the text dataset.

Converts text to lowercase for consistent vocabulary.

### Tokenize Text

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1

Converts words into numbers.

word_index stores vocabulary mapping.

### Create Input Sequences

In [7]:
input_sequences = []

for line in data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

Creates training sequences.

E.g. I love AI

[I, love]

[I, love, AI]

### Padding

In [8]:
max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

Makes all sequences equal length by adding zeros.

E.g.

[0 0 I love]

[0 I love AI]

### Create Training Data

In [9]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

X = input words

y = next word

E.g.
Input : I love

Output: AI

### Build RNN Model

In [10]:
rnn_model = Sequential([
    Embedding(total_words, 50, input_length=max_seq_len-1),
    SimpleRNN(100),
    Dense(total_words, activation='softmax')
])

rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Layers:

Embedding - convert words into vector

SimpleRNN - learns sequence pattern

Dense - predict next word

### Build LSTM Model

In [11]:
lstm_model = Sequential([
    Embedding(total_words, 50, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

Same architecture but LSTM remembers long-term dependencies better than RNN.

### Train Model

In [13]:
early_stop = EarlyStopping(monitor='loss', patience=5)

lstm_model.fit(X, y, epochs=50, callbacks=[early_stop])

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0540 - loss: 5.4998
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.0544 - loss: 5.5149
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.0674 - loss: 5.4450
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0533 - loss: 5.4072
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.0623 - loss: 5.3844
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0557 - loss: 5.3825
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.0674 - loss: 5.2488
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0708 - loss: 5.2618
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.0866 - loss: 5.0866
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0870 - loss: 5.0797
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.0946 - loss: 4.8984
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy:

epochs=50 trains model multiple times.

EarlyStopping stops training when loss stops improving.

### Text Generation Function

In [24]:
def generate_text(seed_text, next_words, model):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        preds = model.predict(token_list)[0]
        predicted = np.random.choice(len(preds), p=preds)

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

Convert input text to tokens

Predict next word

Append predicted word

Repeat until sentence is complete

### Generate Sentence

In [33]:
print(generate_text("deep learning", 10, lstm_model))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
deep learning models are remarkable used patterns in artificial intelligence tools the


## Conclusion

Here we trains an RNN and LSTM model to predict the next word in a sentence. The text is tokenized into sequences, and the model learns patterns between words. After training, a text generation function predicts words sequentially to generate new sentences similar to the training data.